# Stream experiments on saved CLIP features (Colab)

This notebook runs TDA **on features that were already extracted** (e.g. CORe50 on the cluster with
`bash run.sh features core50 ...`). No images and no dataset download are needed: every run only reorders
the saved feature table, so it takes seconds, also on the free Colab GPU.

It is a thin driver of the same files as the cluster: `streams.py` (runs, summary table) and
`analyze_traces.py` (plots of what happens inside the stream). A change pushed to the repository is used
the next time you run the notebook.

**Before the first run**: put the feature files in your Google Drive, in
`MyDrive/TDA_replication/results/features/` (e.g. `core50__RN50.pt`, `core50__ViT-B-16.pt`).
They are in `results/features/` on the cluster account that extracted them (a few hundred MB each).

**How to use it**: *Runtime → Change runtime type → GPU*, edit **Settings**, then *Runtime → Run all*.
Results, traces and plots are saved on Drive (`MyDrive/TDA_replication/results/`), and runs already done
are skipped, so the notebook can be run again after a disconnection.

## 1. GPU (a CPU also works, only slower)

In [ ]:
!nvidia-smi -L || echo "no GPU: runs will use the CPU"

## 2. Settings
- `DATASET`: name of the feature file: `core50` (128x128), `core50_350`, `stream51`, `dtd`, ...
- `CFG_LIKE`: hyper-parameters to use (CORe50 and Stream-51 have no official config). Use the frozen choice of the group.
- `SESSIONS`: CORe50 only, e.g. `"3,7,10"` = official test sessions. `None` = all frames.
- `ORDERS`: `s0,s1,s2,s3` for CORe50; `s0,s1,instance,class_instance,class_iid` for Stream-51.
- `MODES`: `tda` (TDA), `clip` (CLIP alone), `m2` (per-class mean in the cache), `oracle` (cache with true labels), `smooth` (CLIP averaged over the last `WINDOWS` frames).
- `SEEDS`: number of random streams per order. `TRACE = True` saves what happens at every frame (needed for the plots).

In [ ]:
# ======================= SETTINGS =======================
REPO_URL = "https://github.com/gretadell/Visionaries-group-project---TDA.git"
DATASET  = "core50"
BACKBONE = "RN50"                      # or "ViT-B/16"
CFG_LIKE = "caltech101"
SESSIONS = "3,7,10"                    # None for all frames
ORDERS   = "s0,s2,s3"
MODES    = "tda,clip,m2,oracle"
WINDOWS  = "5,15,30"                   # smooth only
SEEDS    = 5
TRACE    = True
# ========================================================

import os
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/TDA_replication"
CODE = "/content/TDA_replication"
os.environ.update({
    "TDA_HOME": CODE,
    "REPO_DIR": f"{CODE}/TDA",                  # official repository (configs, clip package)
    "RESULTS_DIR": f"{DRIVE_ROOT}/results",     # features, runs, traces, plots
})
os.makedirs(f"{DRIVE_ROOT}/results/features", exist_ok=True)
feat_file = f"{DRIVE_ROOT}/results/features/{DATASET}__{BACKBONE.replace('/', '-')}.pt"
print("feature file:", feat_file, "->", "found" if os.path.isfile(feat_file) else "MISSING: copy it there first")

## 3. Code
Downloads the latest version of this repository (any local change in `/content/TDA_replication` is discarded;
nothing on Drive is touched) and the official TDA repository.

In [ ]:
if os.path.isdir(f"{CODE}/.git"):
    !git -C {CODE} fetch -q origin && git -C {CODE} reset -q --hard origin/main
else:
    !git clone -q {REPO_URL} {CODE}
if not os.path.isdir(f"{CODE}/TDA/.git"):
    !git clone -q https://github.com/kdiAAA/TDA.git {CODE}/TDA
!pip -q install ftfy regex
!git -C {CODE} log -1 --format="this repository: %h %cd %s"
%cd {CODE}

## 4. Runs
One run per mode x order x seed. Each prints its accuracy; runs already done are skipped
(with `TRACE = True`, a run done earlier without a trace is redone).

In [ ]:
extra = (f"--sessions {SESSIONS} " if SESSIONS else "") + ("--trace " if TRACE else "")
!python streams.py run --dataset {DATASET} --backbone {BACKBONE} --cfg-like {CFG_LIKE} --order {ORDERS} --mode {MODES} --window {WINDOWS} --repeats {SEEDS} {extra}

## 5. Summary table
Mean accuracy over seeds with a 95% confidence interval, one row per config / frames / mode / order.
Sanity check: the `clip` rows must have the same value for every order (CLIP has no memory).

In [ ]:
!python streams.py summary --dataset {DATASET}

## 6. What happens inside the stream
`analyze_traces.py` reads the traces and writes `summary.md` and five plots in
`MyDrive/TDA_replication/results/analysis/<dataset>__<backbone>/`. The column notes are at the end of `summary.md`.

In [ ]:
frames = f"sessions {SESSIONS}" if SESSIONS else "all"
!python analyze_traces.py --dataset {DATASET} --backbone {BACKBONE} --orders {ORDERS} --config {CFG_LIKE} --frames "{frames}"

import glob
from IPython.display import Image, Markdown, display
out = f"{DRIVE_ROOT}/results/analysis/{DATASET}__{BACKBONE.replace('/', '-')}"
display(Markdown(open(f"{out}/summary.md", encoding="utf-8").read()))
for png in sorted(glob.glob(f"{out}/*.png")):
    print(os.path.basename(png))
    display(Image(png))